# Train YOLO26n on Stop / 35 mph / 45 mph signs

Makes a YOLO26n sign detector that works with your `realtime_camera_detect.py`, `attacks.py` and `attack_eval.py`.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU** (training on CPU would take many hours).
2. Have your **Roboflow API key** ready (free account): https://app.roboflow.com → **Settings** (gear) → **API Keys** → copy the **Private API Key**.

Then **Runtime → Run all**. When it asks, paste your API key and press Enter. Colab downloads the LISA dataset by itself
(https://universe.roboflow.com/kaggle-road-sign-dataset/lisa-bjgh5) - no zip needed.

*Already downloaded the zip?* Upload it to **Google Drive** (drive.google.com → **New → File upload**, into **My Drive**). Then press Enter without a key; the notebook asks to connect your Drive and uses the zip from there. (Colab's own upload button fails on big files.)

**Optional but recommended:** first take pictures of your own signs with `collect_signs.py` and put `my_signs.zip` in Google Drive. Step 2b adds them.

The new model uses the **same class names as your current model** (`35_speed_limit`, `45_speed_limit`, `stop_sign`), so the camera colors, alerts and swap attack keep working.

In [ ]:
!pip -q install -U ultralytics
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

## 1. Get the LISA dataset (API key, or upload a zip)

In [ ]:
from google.colab import files
from getpass import getpass
import zipfile, shutil
from pathlib import Path

RAW = Path('/content/raw_ds')
shutil.rmtree(RAW, ignore_errors=True)
RAW.mkdir(parents=True)

key = getpass('Paste your Roboflow API key and press Enter (or just press Enter to use the zip in your Google Drive): ').strip()
if key:
    !pip -q install roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key=key)
    project = rf.workspace('kaggle-road-sign-dataset').project('lisa-bjgh5')
    VERSION = 2   # the version shown on the Roboflow page ("v2")
    print('Downloading LISA version', VERSION, '... (a few minutes)')
    project.version(VERSION).download('yolov8', location=str(RAW / 'lisa'))  # same folder layout as YOLO26
    del key
else:
    # Big zips (like LISA, ~350 MB) can't go through Colab's upload button ("Unable to read file").
    # So look in your Google Drive first: upload the zip to drive.google.com (My Drive), then run this.
    from google.colab import drive
    drive.mount('/content/drive')
    MY = Path('/content/drive/MyDrive')
    not_mine = lambda p: 'my_signs' not in str(p).lower()                # my_signs is YOUR pictures, not LISA
    found = [z for z in sorted(MY.rglob('LISA*.zip'))] or [z for z in sorted(MY.glob('*.zip')) if not_mine(z)]
    # Unzipped folder in Drive? (a folder with data.yaml + train/ inside, e.g. "LISA.v2i.yolo26")
    folders = [y.parent for y in sorted(MY.rglob('data.yaml')) if (y.parent / 'train').is_dir() and not_mine(y)]
    folders.sort(key=lambda f: 'lisa' not in str(f).lower())        # prefer a LISA folder
    zip_path = None
    if folders:
        src = folders[0]
        n = sum(1 for _ in (src / 'train').rglob('*'))
        print(f'Using unzipped folder from Google Drive: {src}  ({n} files in train/)')
        print('Copying it to Colab so training is fast (a few minutes)...')
        shutil.copytree(src, RAW / src.name)
    elif found:
        zip_path = found[0]
        print('Using zip from Google Drive:', zip_path)
    else:
        raise FileNotFoundError(
            "The LISA dataset isn't in your Google Drive (My Drive).\n"
            "Easiest fix: run this cell again and PASTE YOUR ROBOFLOW API KEY when asked -\n"
            "Colab then downloads LISA by itself (Roboflow > Settings > API Keys > Private key).")
    if zip_path:
        print('Unzipping (about a minute)...')
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(RAW)

yamls = list(RAW.rglob('data.yaml'))
assert yamls, "No data.yaml found - the dataset must be in a YOLO format"
RAW_ROOT = yamls[0].parent
print("Dataset folder:", RAW_ROOT)

## 2. Keep only Stop / 35 / 45

Other sign types (yield, 25 mph, etc.) are dropped. Some pictures with **no** sign are kept as *background* - that teaches the model what "not a sign" looks like, so it boxes your face less.

In [ ]:
import yaml, random, re

NEW_NAMES = ['35_speed_limit', '45_speed_limit', 'stop_sign']        # same as your current model
def target_of(name):
    n = re.sub(r'[^a-z0-9]', '', str(name).lower())
    return {'speedlimit35': 0, '35speedlimit': 0,
            'speedlimit45': 1, '45speedlimit': 1,
            'stop': 2, 'stopsign': 2}.get(n)          # exact match only (not stopAhead, rampSpeedAdvisory35 ...)

cfg = yaml.safe_load(open(RAW_ROOT / 'data.yaml'))
names = cfg['names']
names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
old_to_new = {i: target_of(n) for i, n in enumerate(names)}
print("Using:", {names[i]: NEW_NAMES[t] for i, t in old_to_new.items() if t is not None})
assert sorted(t for t in old_to_new.values() if t is not None) == [0, 1, 2], \
    "This dataset doesn't have all of stop / speedLimit35 / speedLimit45"

OUT = Path('/content/signs3')
shutil.rmtree(OUT, ignore_errors=True)
random.seed(0)
BACKGROUND_SHARE = 0.10
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def split_dirs():
    found = {}
    for split, aliases in [('train', ['train']), ('val', ['valid', 'val']), ('test', ['test'])]:
        for a in aliases:
            if (RAW_ROOT / a / 'images').is_dir():
                found[split] = RAW_ROOT / a
                break
    return found

counts = {}
for split, src in split_dirs().items():
    (OUT / split / 'images').mkdir(parents=True)
    (OUT / split / 'labels').mkdir(parents=True)
    with_sign, background = [], []
    for img in (src / 'images').iterdir():
        if img.suffix.lower() not in IMG_EXT:
            continue
        lab = src / 'labels' / (img.stem + '.txt')
        lines = []
        if lab.is_file():
            for line in lab.read_text().splitlines():
                parts = line.split()
                if len(parts) == 5 and old_to_new.get(int(float(parts[0]))) is not None:
                    lines.append(' '.join([str(old_to_new[int(float(parts[0]))])] + parts[1:]))
        (with_sign if lines else background).append((img, lines))
    n_bg = int(len(with_sign) * BACKGROUND_SHARE / (1 - BACKGROUND_SHARE))
    keep = with_sign + random.sample(background, min(n_bg, len(background)))
    per_class = [0, 0, 0]
    for img, lines in keep:
        shutil.copy(img, OUT / split / 'images' / img.name)
        (OUT / split / 'labels' / (img.stem + '.txt')).write_text('\n'.join(lines))
        for l in lines:
            per_class[int(l.split()[0])] += 1
    counts[split] = dict(images=len(keep), background=len(keep) - len(with_sign),
                         **{NEW_NAMES[i]: per_class[i] for i in range(3)})

# No validation split in the zip? Move 15% of train into one.
if 'val' not in counts:
    (OUT / 'val' / 'images').mkdir(parents=True); (OUT / 'val' / 'labels').mkdir(parents=True)
    imgs = sorted((OUT / 'train' / 'images').iterdir()); random.shuffle(imgs)
    for img in imgs[:int(len(imgs) * 0.15)]:
        shutil.move(img, OUT / 'val' / 'images' / img.name)
        shutil.move(OUT / 'train' / 'labels' / (img.stem + '.txt'), OUT / 'val' / 'labels' / (img.stem + '.txt'))
    counts['val'] = {'images': int(len(imgs) * 0.15), 'note': 'made from train'}

data = {'path': str(OUT), 'train': 'train/images', 'val': 'val/images', 'names': NEW_NAMES}
if (OUT / 'test' / 'images').is_dir():
    data['test'] = 'test/images'
yaml.safe_dump(data, open(OUT / 'data.yaml', 'w'))
for k, v in counts.items():
    print(k, v)

## 2b. Add YOUR own webcam pictures (from `collect_signs.py`)

LISA's signs are filmed from cars and it has few 45 mph signs, so the model struggled with a 45 on your phone.
This adds the pictures you took with `collect_signs.py`: put **`my_signs.zip`** (or the `my_signs` folder) in **Google Drive → My Drive**.

- 80% go into training (each counted **3 times**, so ~450 of your pictures aren't drowned out by ~1,800 LISA pictures)
- 20% are kept aside to test the model **on your own setup** (step 4 shows that score separately)

If nothing is found, training just uses LISA.

In [ ]:
import os
REPEAT = 3          # how many times each of your training pictures is used
MY_VAL_SHARE = 0.2

from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
MY = Path('/content/drive/MyDrive')

MY_RAW = Path('/content/my_signs_raw')
shutil.rmtree(MY_RAW, ignore_errors=True)
MY_RAW.mkdir()
my_zips = sorted(MY.rglob('my_signs*.zip'))
my_dirs = [d for d in sorted(MY.rglob('my_signs*')) if d.is_dir() and list(d.rglob('labels'))]
if my_zips:
    print('Using', my_zips[-1])
    with zipfile.ZipFile(my_zips[-1]) as z:
        z.extractall(MY_RAW)
elif my_dirs:
    print('Using folder', my_dirs[0])
    shutil.copytree(my_dirs[0], MY_RAW / my_dirs[0].name)
else:
    print('No my_signs zip/folder in My Drive - training on LISA only.')

lab_dirs = [d for d in MY_RAW.rglob('labels') if (d.parent / 'images').is_dir()]
HAVE_MINE = bool(lab_dirs)
if (my_zips or my_dirs) and not HAVE_MINE:
    print('Found my_signs, but no images/ + labels/ folders inside it. What Colab sees:')
    for p in sorted(MY_RAW.rglob('*'))[:20]:
        print('   ', p.relative_to(MY_RAW))
    raise RuntimeError("my_signs has no images/ and labels/ inside. If you just uploaded it, Google Drive may "
                       "still be syncing: wait 2-3 minutes, then run THIS cell again (not Run all).")
if HAVE_MINE:
    src = lab_dirs[0].parent
    pairs = []
    for img in sorted((src / 'images').iterdir()):
        lab = src / 'labels' / (img.stem + '.txt')
        if img.suffix.lower() in IMG_EXT and lab.is_file():
            pairs.append((img, lab))
    if len(pairs) < 50:
        raise RuntimeError(f"Only {len(pairs)} of your pictures have labels. If you just uploaded my_signs, "
                           "Google Drive may still be syncing: wait 2-3 minutes, then run THIS cell again.")
    random.seed(1)
    random.shuffle(pairs)
    n_val = max(1, int(len(pairs) * MY_VAL_SHARE))
    my_val, my_train = pairs[:n_val], pairs[n_val:]
    for d in ['myval/images', 'myval/labels']:
        (OUT / d).mkdir(parents=True, exist_ok=True)
    per = [0, 0, 0, 0]          # 35, 45, stop, background
    for img, lab in my_train:
        for r in range(REPEAT):
            stem = f'mine_{img.stem}_r{r}'
            shutil.copy(img, OUT / 'train/images' / (stem + img.suffix))
            shutil.copy(lab, OUT / 'train/labels' / (stem + '.txt'))
        t = lab.read_text().strip()
        per[int(t.split()[0]) if t else 3] += 1
    for img, lab in my_val:
        for split in ['val', 'myval']:
            shutil.copy(img, OUT / split / 'images' / ('mine_' + img.name))
            shutil.copy(lab, OUT / split / 'labels' / ('mine_' + img.stem + '.txt'))
    yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': 'myval/images', 'names': NEW_NAMES},
                   open(OUT / 'myval.yaml', 'w'))
    print(f'Your pictures: {len(my_train)} for training (x{REPEAT}), {len(my_val)} kept for testing')
    print(f'  training: 35={per[0]}  45={per[1]}  stop={per[2]}  background={per[3]}')
    if min(per[:3]) < 30:
        print('  NOTE: some sign has fewer than 30 pictures - collect more of it for best results.')

## 3. Train YOLO26n

About 45-60 minutes on a T4. It stops early if it stops improving.

**Progress is saved to your Google Drive** (`My Drive/yolo_runs/signs_yolo26n`) after every round. If Colab disconnects, just open the notebook again and **Run all**: it rebuilds the dataset and **continues from where it stopped** instead of starting over.

In [ ]:
from ultralytics import YOLO
import os
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/yolo_runs'          # saved in Google Drive -> survives disconnects
RUN_DIR = Path(PROJECT) / 'signs_yolo26n'
last = RUN_DIR / 'weights' / 'last.pt'

if last.is_file():
    print('Found earlier progress in Google Drive - continuing from', last)
    try:
        YOLO(str(last)).train(resume=True)
    except Exception as e:
        if 'finished' in str(e).lower() or 'nothing to resume' in str(e).lower():
            print('That training run already finished - using its result.')
        else:
            raise
else:
    YOLO('yolo26n.pt').train(
        data='/content/signs3/data.yaml',
        epochs=60,
        imgsz=640,
        batch=16,
        patience=15,
        fliplr=0.0,        # never mirror: a mirrored "35" or "STOP" is not a real sign
        project=PROJECT,
        name='signs_yolo26n',
        exist_ok=True,
    )

## 4. Check how good it is

In [ ]:
best = Path('/content/drive/MyDrive/yolo_runs/signs_yolo26n/weights/best.pt')
if not best.is_file():
    best = sorted(Path('/content').rglob('signs_yolo26n/weights/best.pt'))[-1]
print('Using', best)
m = YOLO(str(best))

def report(title, yaml_file):
    met = m.val(data=yaml_file, split='val', verbose=False)
    print(f'\n=== {title} ===')
    print('mAP50 (all signs):', round(float(met.box.map50), 3), ' <- 1.0 is perfect; 0.8+ is good')
    for i, n in enumerate(NEW_NAMES):
        print(f'  {n:15s} mAP50-95 = {met.box.maps[i]:.3f}')

report('All test pictures (LISA + yours)', '/content/signs3/data.yaml')
if HAVE_MINE:
    report('YOUR webcam pictures only  <- the score that matters for the live camera', '/content/signs3/myval.yaml')

## 5. Download your new model

You get **`best_signs_yolo26n.pt`**. Put it in **New folder (4)**. Because its name contains "best", the scripts pick it **automatically** over your old model. To use the old one again, run with `--weights "Detecting stop signs traffic (3).pt"`.

If the download doesn't start, it's also in Google Drive: **My Drive → yolo_runs → signs_yolo26n → weights → best.pt**. Download that and rename it `best_signs_yolo26n.pt`.

In [ ]:
out = Path('/content/best_signs_yolo26n.pt')
shutil.copy(best, out)
files.download(str(out))